# Figure 18 — what σ_f does: how far the belief can travel from the data

Same six observations, same length-scale, same noise. Only the signal standard deviation σ_f changes.

σ_f is the **prior** spread of the function, so it sets the CEILING of the posterior band: far from any data, σ(x) → σ_f. At the observations it does almost nothing — all three panels fit the data equally well (mean |error| 0.017, 0.001, 0.000). The whole effect is in the GAPS, and therefore entirely on how much the campaign wants to explore.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

style.SHOW_TEXT = True   # False -> clean panels for ungrouping into pptx shapes

xs = np.linspace(0, 10, 600)
OX = np.array([1.15, 2.90, 4.30, 6.10, 7.40, 8.90])
OY = land.f1d(OX)
LS, SN = 0.85, 0.05          # held fixed throughout

SETTINGS = [
    (0.25, "too small", style.RED,
     "claims to know the gaps\n→ BO stops exploring"),
    (1.00, "about right", style.INK,
     "confident at the data,\nhonestly ignorant between it"),
    (3.00, "too large", style.GOLD,
     "everything unmeasured looks wide open\n→ BO explores forever"),
]

fig, axes = plt.subplots(1, 3, figsize=(style.FIG_W_FULL, 3.5), sharey=True,
                         gridspec_kw=dict(wspace=0.08))

for ax, (sf, tag, colour, note) in zip(axes, SETTINGS):
    g = gpmod.GP(gpmod.matern52, ls=LS, sf=sf, sn=SN).fit(OX[:, None], OY)
    mu, sd = g.predict(xs[:, None])

    # the ceiling: sigma(x) can never exceed sigma_f
    ax.axhline(2 * sf, color=colour, ls=":", lw=1.1, alpha=0.85)
    ax.axhline(-2 * sf, color=colour, ls=":", lw=1.1, alpha=0.85)
    style.text(ax, 9.85, 2 * sf + 0.15, r"$\pm 2\sigma_f$ ceiling",
               ha="right", va="bottom", fontsize=9.5, color=colour,
               bbox=dict(fc="white", alpha=0.85, ec="none", pad=1.6))

    ax.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL,
                    alpha=0.16, lw=0)
    ax.plot(xs, mu, color=style.TEAL, lw=2.0, zorder=5)
    ax.plot(xs, land.f1d(xs), "--", color=style.INK, lw=1.0, alpha=0.40, zorder=4)
    ax.plot(OX, OY, "o", ms=6.5, color=style.RED, mec="white", mew=1.1, zorder=7)

    ax.set_ylim(-6.6, 6.6)
    style.xlabel(ax, "reaction parameter  x")
    style.title(ax, fr"$\sigma_f$ = {sf:.2f}  —  {tag}", loc="left",
                fontsize=11.5, color=colour, pad=8)
    style.text(ax, 0.03, 0.03, note, transform=ax.transAxes, fontsize=8.8,
               color=style.INK, va="bottom",
               bbox=dict(fc="white", alpha=0.85, ec="none", pad=2.0))

    # how badly does the mean miss its own observations?
    mu_at, _ = g.predict(OX[:, None])
    miss = np.abs(mu_at - OY).mean()
    gap = sd[np.argmin(np.abs(xs - 5.2))]     # sd in the widest gap
    style.text(ax, 0.03, 0.965,
               f"fit error at the data   {miss:.3f}\nsd in the gap   {gap:.2f}",
               transform=ax.transAxes, ha="left", va="top", fontsize=9.2,
               color=colour, fontweight="bold",
               bbox=dict(fc="white", alpha=0.88, ec="none", pad=2.2))
    print(f"sigma_f={sf:<5} mean |fit error| at the data = {miss:.3f} "
          f"| sd in the gap at x=5.2 = {gap:.3f}  (ceiling {sf:.2f})")

style.ylabel(axes[0], "yield (arb.)")
style.text(axes[1], 0.5, 1.10, "dashed = the truth   ·   dots = experiments   ·   "
           r"$\ell$ and $\sigma_n$ identical in all three panels",
           transform=axes[1].transAxes, ha="center", fontsize=9.5,
           color=style.GRAY)
style.save(fig, "fig_18_sigma_f", OUT)